# Chương 4, notebook 6: RNN, LSTM, GRU và MLP viết bằng NumPy thuần
Thư viện `rnn_numpy.py`: lan truyền tiến, lan truyền ngược theo thời gian (BPTT), Adam, cắt chuẩn gradient, dừng sớm.
Trọng số bố trí giống `torch.nn`, nên có thể nạp cùng trọng số vào PyTorch để đối chiếu từng con số.

In [1]:
import os, sys
while not os.path.exists("rnn_numpy.py") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")          # di len den thu muc co rnn_numpy.py
sys.path.insert(0, os.getcwd())
import numpy as np
import rnn_numpy as R

## 1. Kiểm chứng với PyTorch và sai phân hữu hạn

In [2]:
exec(open("verify_numpy.py", encoding="utf-8").read())

RNN forward diff vs torch 8.33e-17 | grad diff vs torch 8.33e-17 | finite-diff rel err 1.46e-09 | params 113
GRU forward diff vs torch 1.11e-16 | grad diff vs torch 2.78e-17 | finite-diff rel err 2.19e-07 | params 321
LSTM forward diff vs torch 4.16e-17 | grad diff vs torch 2.08e-17 | finite-diff rel err 2.24e-08 | params 425


## 2. Huấn luyện nhanh một hạt giống trên Churn
Chạy đủ ba hạt giống và cả bài AMZN bằng `python run_numpy.py churn|ret|vol`.

In [3]:
import lab06_data as D
C = D.churn_splits(); hp = D.CHURN_HP
seq = lambda X: X[:, :, None]
data = dict(X_train=seq(C["X_train"]), y_train=C["y_train"], X_val=seq(C["X_val"]), y_val=C["y_val"])
for cell in ("RNN", "LSTM", "GRU"):
    m = R.SeqNetNP(cell, 1, hp["hidden"], 0.0, 42)
    info = R.fit(m, data, R.bce_grad, hp, 42)
    pt = R.sigm(m.forward(seq(C["X_test"]).astype(np.float64)))
    thr = D.best_threshold(C["y_val"], R.sigm(m.forward(seq(C["X_val"]).astype(np.float64))))
    print(cell, m.n_params(), "tham số", D.cls_metrics(C["y_test"], pt, thr)["roc_auc"], "ROC-AUC", info["epochs"], "epoch", info["seconds"], "giây")

RNN 1153 tham số 0.8553 ROC-AUC 35 epoch 5.13 giây


LSTM 4513 tham số 0.8465 ROC-AUC 32 epoch 17.17 giây


GRU 3393 tham số 0.8334 ROC-AUC 31 epoch 11.37 giây


## 3. Kết quả đầy đủ (ba hạt giống) đã lưu

In [4]:
import json, glob
for f in sorted(glob.glob("results/numpy__churn__*.json")):
    d = json.load(open(f, encoding="utf-8")); print(d["model"], d["params"], d["summary"]["roc_auc"])

GRU 3393 [0.8456, 0.00992068546018872]
LSTM 4513 [0.8407333333333334, 0.004616155206325801]
MLP 417 [0.8573333333333334, 0.0005734883511361592]
RNN 1153 [0.8549666666666668, 0.0011671427600007333]
